In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("Understanding DAG Plan")\
        .master("local[8]")\
        .getOrCreate()

spark

In [7]:
# Check defaultparallesism

spark.sparkContext.defaultParallelism

8

In [9]:
# Disable AQE and Brodacast join

spark.conf.set("spark.sql.adaptive.enabled",False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled",False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",-1)

In [10]:
# create dataframe

df_1 = spark.range(4,200,2)
df_2 = spark.range(2,200,4)


In [11]:
# Repartition data
df_3 = df_1.repartition(5)
df_4 = df_2.repartition(7)

In [15]:
df_3.rdd.getNumPartitions()


5

In [16]:
df_4.rdd.getNumPartitions()

7

In [12]:
df_join = df_3.join(df_4,on="id")

In [17]:
df_sum = df_join.selectExpr("sum(id) as total_sum")

In [18]:
# view data
df_sum.show()

+---------+
|total_sum|
+---------+
|     4998|
+---------+



In [19]:
df_sum.explain()

== Physical Plan ==
*(6) HashAggregate(keys=[], functions=[sum(id#4L)])
+- Exchange SinglePartition, ENSURE_REQUIREMENTS, [id=#174]
   +- *(5) HashAggregate(keys=[], functions=[partial_sum(id#4L)])
      +- *(5) Project [id#4L]
         +- *(5) SortMergeJoin [id#4L], [id#6L], Inner
            :- *(2) Sort [id#4L ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(id#4L, 200), ENSURE_REQUIREMENTS, [id=#158]
            :     +- Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [id=#157]
            :        +- *(1) Range (4, 200, step=2, splits=8)
            +- *(4) Sort [id#6L ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(id#6L, 200), ENSURE_REQUIREMENTS, [id=#165]
                  +- Exchange RoundRobinPartitioning(7), REPARTITION_BY_NUM, [id=#164]
                     +- *(3) Range (2, 200, step=4, splits=8)




In [20]:
df_union = df_sum.union(df_4)

In [21]:
df_union.show()

+---------+
|total_sum|
+---------+
|     4998|
|       14|
|       38|
|       50|
|       74|
|      110|
|      130|
|      154|
|      186|
|       10|
|       30|
|       54|
|       98|
|      118|
|      138|
|      158|
|      178|
|        6|
|       42|
|       70|
+---------+
only showing top 20 rows

